# Étape 4 — Comparaison de modèles Ollama (`llama3.2:3b` vs `llama3`)

**Objectif de ce notebook :**
- Poser le même jeu de questions aux deux modèles
- Mesurer le temps de réponse de chacun
- Comparer la qualité des réponses côte à côte, pour choisir objectivement lequel
  utiliser (ou documenter le compromis vitesse/qualité dans le rapport)

**Prérequis :** les deux modèles doivent être téléchargés :
```
ollama list
```
doit afficher à la fois `llama3.2:3b` et `llama3`.

**Kernel à utiliser :** `Python (RAG GED)`.

## 0. Imports et accès au dossier `src/`

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

from rag_chain import RagAssistant

## 1. Jeu de questions de test

Un mélange volontaire de cas déjà rencontrés (question factuelle précise, question de
suivi, question hors-sujet) pour bien voir les différences de comportement, pas
seulement de vitesse.

In [2]:
QUESTIONS_TEST = [
    "Comment fonctionne l'atelier Préparation ?",
    "Quel logiciel est utilisé pour CMIM ?",
    "Quelle est la différence entre PEC et RD ?",
    "C'est quoi la différence entre GED et archivage électronique ?",
    "Quel est le salaire moyen des opérateurs GED ?",
]

print(f"{len(QUESTIONS_TEST)} questions de test définies.")

5 questions de test définies.


## 2. Fonction d'évaluation d'un modèle

Charge l'assistant avec le modèle donné, pose chaque question (sans historique, pour
une comparaison à conditions égales), mesure le temps de réponse.

In [3]:
def evaluer_modele(nom_modele: str) -> list[dict]:
    print(f"\n=== Chargement de l'assistant avec le modèle : {nom_modele} ===")
    assistant = RagAssistant(model=nom_modele)

    resultats = []
    for question in QUESTIONS_TEST:
        debut = time.time()
        reponse = assistant.ask(question)
        duree = time.time() - debut

        resultats.append({
            "modele": nom_modele,
            "question": question,
            "reponse": reponse["answer"],
            "temps_secondes": round(duree, 1),
        })
        print(f"  [{duree:.1f}s] {question}")

    return resultats

## 3. Exécution — `llama3.2:3b`

⏱️ Compte quelques dizaines de secondes au total (le modèle est déjà téléchargé et
rapide).

In [4]:
resultats_3b = evaluer_modele("llama3.2:3b")


=== Chargement de l'assistant avec le modèle : llama3.2:3b ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [35.7s] Comment fonctionne l'atelier Préparation ?
  [24.1s] Quel logiciel est utilisé pour CMIM ?
  [38.5s] Quelle est la différence entre PEC et RD ?
  [41.5s] C'est quoi la différence entre GED et archivage électronique ?
  [30.8s] Quel est le salaire moyen des opérateurs GED ?


## 4. Exécution — `llama3` (8B)

⏱️ Nettement plus lent (modèle ~3x plus gros). Assure-toi qu'il est bien téléchargé
(`ollama list`) avant de lancer cette cellule.

In [5]:
resultats_8b = evaluer_modele("llama3")


=== Chargement de l'assistant avec le modèle : llama3 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [211.6s] Comment fonctionne l'atelier Préparation ?
  [51.4s] Quel logiciel est utilisé pour CMIM ?
  [69.0s] Quelle est la différence entre PEC et RD ?
  [77.9s] C'est quoi la différence entre GED et archivage électronique ?
  [51.3s] Quel est le salaire moyen des opérateurs GED ?


## 5. Tableau comparatif des temps de réponse

In [6]:
df = pd.DataFrame(resultats_3b + resultats_8b)

tableau_temps = df.pivot(index="question", columns="modele", values="temps_secondes")
tableau_temps["difference_secondes"] = (
    tableau_temps["llama3"] - tableau_temps["llama3.2:3b"]
)
tableau_temps

modele,llama3,llama3.2:3b,difference_secondes
question,,,
C'est quoi la différence entre GED et archivage électronique ?,77.9,41.5,36.4
Comment fonctionne l'atelier Préparation ?,211.6,35.7,175.9
Quel est le salaire moyen des opérateurs GED ?,51.3,30.8,20.5
Quel logiciel est utilisé pour CMIM ?,51.4,24.1,27.3
Quelle est la différence entre PEC et RD ?,69.0,38.5,30.5


In [7]:
moyenne_3b = df[df["modele"] == "llama3.2:3b"]["temps_secondes"].mean()
moyenne_8b = df[df["modele"] == "llama3"]["temps_secondes"].mean()

print(f"Temps de réponse moyen — llama3.2:3b : {moyenne_3b:.1f}s")
print(f"Temps de réponse moyen — llama3      : {moyenne_8b:.1f}s")
print(f"llama3 est {moyenne_8b / moyenne_3b:.1f}x plus lent en moyenne")

Temps de réponse moyen — llama3.2:3b : 34.1s
Temps de réponse moyen — llama3      : 92.2s
llama3 est 2.7x plus lent en moyenne


## 6. Comparaison des réponses côte à côte

Lis attentivement chaque paire de réponses : laquelle est la plus précise, la plus
fidèle aux documents, la moins sujette aux erreurs vues précédemment (mélange
d'informations, recopie de la structure du document) ?

In [8]:
for question in QUESTIONS_TEST:
    print("=" * 90)
    print(f"QUESTION : {question}")
    print("=" * 90)

    reponse_3b = next(r for r in resultats_3b if r["question"] == question)
    reponse_8b = next(r for r in resultats_8b if r["question"] == question)

    print(f"\n--- llama3.2:3b ({reponse_3b['temps_secondes']}s) ---")
    print(reponse_3b["reponse"])

    print(f"\n--- llama3 ({reponse_8b['temps_secondes']}s) ---")
    print(reponse_8b["reponse"])
    print()

QUESTION : Comment fonctionne l'atelier Préparation ?

--- llama3.2:3b (35.7s) ---
1. L'atelier Préparation consiste à ouvrir chaque dossier du lot, 
2. à classer les pièces à l'intérieur du dossier dans le bon ordre, 
3. à vérifier la complétude du dossier (toutes les pièces attendues sont présentes), 
4. et à retirer les agrafes et trombones qui pourraient endommager le scanner.

--- llama3 (211.6s) ---
L'atelier Préparation consiste à ouvrir chaque dossier du lot, le classer, vérifier la complétude des pièces justificatives et retirer les agrafes et trombones qui pourraient endommager le scanner.

QUESTION : Quel logiciel est utilisé pour CMIM ?

--- llama3.2:3b (24.1s) ---
Kofax Express.

--- llama3 (51.4s) ---
Kofax Express.

QUESTION : Quelle est la différence entre PEC et RD ?

--- llama3.2:3b (38.5s) ---
PEC (Prise En Charge) et RD (Refus de Dossier) sont deux types de décisions possibles à l'issue du traitement d'un dossier. La PEC signifie que le dossier est accepté et pris e

## 7. Exporter le tableau pour le rapport

Génère un fichier Markdown avec le tableau des temps, réutilisable directement dans
le chapitre 4 (justification du choix du modèle).

In [ ]:
export_path = PROJECT_ROOT / "comparaison_modeles.md"

with open(export_path, "w", encoding="utf-8") as f:
    f.write("# Comparaison llama3.2:3b vs llama3\n\n")
    f.write(f"Temps de réponse moyen sur {len(QUESTIONS_TEST)} questions :\n\n")
    f.write(f"- **llama3.2:3b** : {moyenne_3b:.1f} secondes\n")
    f.write(f"- **llama3** : {moyenne_8b:.1f} secondes\n\n")
    f.write("## Détail par question\n\n")
    f.write(tableau_temps.to_markdown())
    f.write("\n")

print(f"Exporté vers : {export_path}")

Exporté vers : c:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\comparaison_modeles.md


: 

## Conclusion

Utilise les résultats ci-dessus pour trancher :
- Si `llama3` corrige nettement les erreurs observées avec `llama3.2:3b` (mélange
  d'infos, recopie de structure) **et** que le temps de réponse reste acceptable pour
  un usage réel par les opérateurs → passe `OLLAMA_MODEL = "llama3"` dans
  `src/rag_chain.py`.
- Si la différence de qualité est faible mais l'écart de vitesse important → garde
  `llama3.2:3b`, et documente ce compromis vitesse/qualité dans ton rapport (bon
  argument technique à valoriser).

Le fichier `comparaison_modeles.md` généré ci-dessus peut être intégré tel quel (ou
adapté) dans le chapitre 4 du rapport.